In [ ]:
# Cell 1: Imports

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)

print("Libraries imported successfully.")

In [ ]:
# Cell 2: Project Configuration

SEED = 42
N_SAMPLES = 768

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

FEATURES = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age"
]

print("Project configuration completed.")

In [ ]:
# Cell 3: Generate Synthetic Dataset

def generate_dataset(n=N_SAMPLES, seed=SEED):
    rng = np.random.default_rng(seed)

    pregnancies = np.clip(
        rng.poisson(2.4, n), 0, 15
    ).astype(int)

    age = np.clip(
        rng.normal(40.7, 14.3, n),
        21, 81
    ).round().astype(int)

    glucose = np.clip(
        rng.normal(
            109.5 + 0.55 * (age - 40),
            27.0,
            n
        ),
        0, 191
    ).round().astype(int)

    bmi = np.clip(
        rng.normal(
            29.3 + 0.10 * (glucose - 110) / 5,
            7.0,
            n
        ),
        0, 48.4
    ).round(1)

    blood_pressure = np.clip(
        rng.normal(
            66.1 + 0.18 * (bmi - 29),
            17.5,
            n
        ),
        0, 111
    ).round().astype(int)

    skin_thickness = np.clip(
        rng.normal(
            20 + 0.7 * (bmi - 29),
            9.5,
            n
        ),
        0, 55
    ).round().astype(int)

    insulin = np.clip(
        rng.normal(
            71 + 1.4 * (glucose - 110)
            + 0.8 * (bmi - 29),
            65,
            n
        ),
        0, 268
    ).round().astype(int)

    pedigree = np.clip(
        rng.lognormal(
            mean=np.log(0.43),
            sigma=0.55,
            size=n
        ),
        0.08,
        2.10
    ).round(3)

    insulin_mismatch = np.maximum(
        glucose - (insulin * 0.20 + 70),
        0
    )

    risk_score = (
        -4.15
        + 0.030 * (glucose - 100)
        + 0.075 * (bmi - 25)
        + 0.028 * (age - 35)
        + 0.32 * pregnancies
        + 0.95 * (pedigree - 0.4)
        + 0.010 * (blood_pressure - 65)
        + 0.006 * insulin_mismatch
    )

    probability = 1 / (1 + np.exp(-risk_score))
    outcome = rng.binomial(1, probability, n)

    return pd.DataFrame({
        "Pregnancies": pregnancies,
        "Glucose": glucose,
        "BloodPressure": blood_pressure,
        "SkinThickness": skin_thickness,
        "Insulin": insulin,
        "BMI": bmi,
        "DiabetesPedigreeFunction": pedigree,
        "Age": age,
        "Outcome": outcome
    })

df = generate_dataset()

print("Dataset generated successfully.")
print("Shape:", df.shape)

In [ ]:
# Cell 4: Display Dataset

df.head()

In [ ]:
# Cell 5: Save Dataset

df.to_csv("diabetes.csv", index=False)
print("diabetes.csv saved successfully.")

In [ ]:
# Cell 6: Dataset Information

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

In [ ]:
# Cell 7: Statistical Summary

df.describe()

In [ ]:
# Cell 8: Missing Values

print(df.isnull().sum())

In [ ]:
# Cell 9: Duplicate Records

print("Duplicate records:", df.duplicated().sum())

In [ ]:
# Cell 10: Outcome Distribution

print(df["Outcome"].value_counts())
print("\nPercentage distribution:")
print(df["Outcome"].value_counts(normalize=True) * 100)

In [ ]:
# Cell 11: Outcome Distribution Plot

plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="Outcome")
plt.title("Diabetes Outcome Distribution")
plt.xlabel("Outcome (0 = Non-Diabetic, 1 = Diabetic)")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_distribution.png", dpi=300)
plt.show()

In [ ]:
# Cell 12: Feature Histograms

df.hist(figsize=(14, 10), bins=20)
plt.suptitle("Feature Distributions", fontsize=16)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_histograms.png", dpi=300)
plt.show()

In [ ]:
# Cell 13: Identify Invalid Zero Values

invalid_zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

for column in invalid_zero_columns:
    print(f"{column}: {(df[column] == 0).sum()} invalid zero values")

In [ ]:
# Cell 14: Replace Invalid Zeros

def replace_invalid_zeros(data):
    data = data.copy()

    for column in invalid_zero_columns:
        data[column] = data[column].replace(0, np.nan)

    return data

cleaned_df = replace_invalid_zeros(df)

print("Invalid zero values replaced with NaN.")

In [ ]:
# Cell 15: Missing Values After Cleaning

print(cleaned_df.isnull().sum())

In [ ]:
# Cell 16: Correlation Heatmap

plt.figure(figsize=(10, 7))
sns.heatmap(
    cleaned_df.corr(numeric_only=True),
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "correlation_heatmap.png", dpi=300)
plt.show()

In [ ]:
# Cell 17: Define Features and Target

X = cleaned_df[FEATURES]
y = cleaned_df["Outcome"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# Cell 18: Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

In [ ]:
# Cell 19: SVM Model

svm_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf",
        C=1.5,
        gamma="scale",
        probability=True,
        random_state=SEED
    ))
])

print("SVM model created.")

In [ ]:
# Cell 20: Gaussian Naive Bayes Model

gnb_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", GaussianNB())
])

print("Gaussian Naive Bayes model created.")

In [ ]:
# Cell 21: Decision Tree Model

tree_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=12,
        class_weight="balanced",
        random_state=SEED
    ))
])

print("Decision Tree model created.")

In [ ]:
# Cell 22: Store Models

models = {
    "SVM": svm_model,
    "Gaussian Naive Bayes": gnb_model,
    "Decision Tree": tree_model
}

print("Models:")
for name in models:
    print("-", name)

In [ ]:
# Cell 23: Train Models

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} training completed.")

In [ ]:
# Cell 24: Generate Predictions

predictions = {}
probabilities = {}

for name, model in models.items():
    predictions[name] = model.predict(X_test)
    probabilities[name] = model.predict_proba(X_test)[:, 1]

print("Predictions generated successfully.")

In [ ]:
# Cell 25: Calculate Evaluation Metrics

results = []

for name in models:
    y_pred = predictions[name]
    y_prob = probabilities[name]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)

In [ ]:
# Cell 26: Display Model Results

results_df

In [ ]:
# Cell 27: Save Model Results

results_df.to_csv(
    OUTPUT_DIR / "model_results.csv",
    index=False
)

print("Model results saved.")

In [ ]:
# Cell 28: Confusion Matrices

for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Non-Diabetic", "Diabetic"],
        yticklabels=["Non-Diabetic", "Diabetic"]
    )

    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()

    filename = (
        name.lower()
        .replace(" ", "_")
        .replace("-", "")
    )

    plt.savefig(
        OUTPUT_DIR / f"{filename}_confusion_matrix.png",
        dpi=300
    )

    plt.show()

In [ ]:
# Cell 29: ROC Curve Comparison

plt.figure(figsize=(8, 6))

for name, y_prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)

    plt.plot(
        fpr,
        tpr,
        label=f"{name} (AUC = {auc:.4f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "roc_curve_comparison.png",
    dpi=300
)

plt.show()

In [ ]:
# Cell 30: Model Performance Comparison

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "ROC-AUC"
]

ax = (
    results_df
    .set_index("Model")[metrics]
    .plot(
        kind="bar",
        figsize=(11, 6),
        ylim=(0, 1)
    )
)

ax.set_title("Model Performance Comparison")
ax.set_ylabel("Score")
ax.set_xlabel("Model")

plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "model_comparison.png",
    dpi=300
)

plt.show()

In [ ]:
# Cell 31: Best Model

best_model_name = (
    results_df
    .loc[
        results_df["F1-Score"].idxmax(),
        "Model"
    ]
)

print("Best model based on F1-score:")
print(best_model_name)

In [ ]:
# Cell 32: Decision Tree Visualization

tree_classifier = tree_model.named_steps["classifier"]

plt.figure(figsize=(20, 10))

plot_tree(
    tree_classifier,
    feature_names=FEATURES,
    class_names=["Non-Diabetic", "Diabetic"],
    filled=True,
    max_depth=4,
    rounded=True,
    fontsize=8
)

plt.title("Decision Tree - Top Four Levels")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "decision_tree.png",
    dpi=300
)

plt.show()

In [ ]:
# Cell 33: Decision Tree Feature Importance

importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": tree_classifier.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

importance

In [ ]:
# Cell 34: Feature Importance Plot

plt.figure(figsize=(10, 6))

sns.barplot(
    data=importance,
    x="Importance",
    y="Feature"
)

plt.title("Decision Tree Feature Importance")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "feature_importance.png",
    dpi=300
)

plt.show()

In [ ]:
# Cell 35: Sample Patient

sample_patient = pd.DataFrame([{
    "Pregnancies": 4,
    "Glucose": 156,
    "BloodPressure": 78,
    "SkinThickness": 34,
    "Insulin": 165,
    "BMI": 34.8,
    "DiabetesPedigreeFunction": 0.72,
    "Age": 46
}])

sample_patient

In [ ]:
# Cell 36: Decision Tree Sample Prediction

sample_prediction = tree_model.predict(sample_patient)[0]
sample_probability = tree_model.predict_proba(sample_patient)[0, 1]

prediction_label = (
    "Diabetic"
    if sample_prediction == 1
    else "Non-Diabetic"
)

print("Sample Patient Prediction")
print("=========================")
print("Prediction:", prediction_label)
print("Estimated probability:", round(sample_probability, 4))

In [ ]:
# Cell 37: Compare All Models on Sample Patient

for name, model in models.items():
    prediction = model.predict(sample_patient)[0]
    probability = model.predict_proba(sample_patient)[0, 1]

    label = (
        "Diabetic"
        if prediction == 1
        else "Non-Diabetic"
    )

    print(
        f"{name}: {label} "
        f"(Probability = {probability:.4f})"
    )

In [ ]:
# Cell 38: Final Summary

print("=" * 60)
print("FINAL MODEL COMPARISON")
print("=" * 60)

display(results_df)

print("\nBest Model by F1-score:")
print(best_model_name)

print("\nProject completed successfully.")

print(
    "\nNOTE: This project uses a synthetic dataset "
    "for academic/educational purposes."
)

print(
    "It is NOT a clinical diagnostic system."
)